In [1]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,016 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-se

In [2]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [3]:
!ollama pull llama3

In [4]:
model_name = "llama3"

In [5]:
!pip install ollama pandas tqdm

In [6]:
import os

REPO_URL = "https://github.com/simona-wang/negotiation_arena.git"
PROJECT_DIR = "/content/negotiation_arena"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Results directory:", RESULTS_DIR)

Cloning into '/content/negotiation_arena'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 210 (delta 136), reused 193 (delta 119), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 1.07 MiB | 16.53 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Project directory: /content/negotiation_arena
Results directory: /content/negotiation_arena/results


In [7]:
import pandas as pd
import ollama
from tqdm import tqdm

In [8]:
scenarios = [
    {
        "scenario_id": 1,
        "domain": "job_contract_negotiation"
    },
    {
        "scenario_id": 2,
        "domain": "job_contract_negotiation"
    },
    {
        "scenario_id": 3,
        "domain": "job_contract_negotiation"
    }
]

In [9]:
conditions = [
    {
        "condition_name": "cooperative",
        "candidate_style": "cooperative",
        "employer_style": "cooperative"
    },
    {
        "condition_name": "competitive",
        "candidate_style": "competitive",
        "employer_style": "competitive"
    },
    {
        "condition_name": "mixed",
        "candidate_style": "competitive",
        "employer_style": "cooperative"
    }
]

In [10]:
def build_candidate_prompt(style):
    return f"""
You are the Candidate in a job contract negotiation.

Private constraints:
- Target salary: 90,000 USD
- Minimum acceptable salary: 85,000 USD
- Preferred working hours: 8 hours
- Maximum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never accept salary below 85,000 USD.
- Never accept working hours above 9.
- If the employer offers salary >= 85,000 and working hours <= 9, explicitly accept.
- If you accept, write DECISION: accept.
- If the offer is not acceptable, write DECISION: continue.
- If no agreement seems possible, write DECISION: quit.
- Reply concisely.
- Do not simulate both speakers. Reply only as Candidate.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [11]:
def build_employer_prompt(style):
    return f"""
You are the Employer in a job contract negotiation.

Private constraints:
- Preferred salary offer: 75,000 USD
- Maximum salary offer: 87,000 USD
- Preferred working hours: 10 hours
- Minimum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never offer more than 87,000 USD.
- Never accept working hours below 9.
- If the candidate proposes salary <= 87,000 and working hours >= 9, explicitly accept.
- If you accept, write DECISION: accept.
- If the proposal is not acceptable, write DECISION: continue.
- If no agreement seems possible, write DECISION: quit.
- Reply concisely.
- Do not simulate both speakers. Reply only as Employer.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [12]:
def generate_response(system_prompt, user_message):
    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    return response["message"]["content"].strip()

In [13]:
test = generate_response(
    "You are a concise assistant.",
    "Reply with: OK"
)

print(test)

OK


In [14]:
import re

def parse_structured_response(text):
    result = {
        "message": None,
        "salary_offer": None,
        "hours_offer": None,
        "decision": None
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("MESSAGE:"):
            result["message"] = line.replace("MESSAGE:", "").strip()

        elif line.startswith("SALARY_OFFER:"):
            value = line.replace("SALARY_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["salary_offer"] = None
            else:
                number = re.search(r"\d[\d,]*", value)
                if number:
                    result["salary_offer"] = int(number.group().replace(",", ""))

        elif line.startswith("HOURS_OFFER:"):
            value = line.replace("HOURS_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["hours_offer"] = None
            else:
                number = re.search(r"\d+(\.\d+)?", value)
                if number:
                    result["hours_offer"] = float(number.group())

        elif line.startswith("DECISION:"):
            result["decision"] = line.replace("DECISION:", "").strip().lower()

    return result

In [15]:
def is_valid_agreement_for_candidate(salary, hours):
    return (
        salary is not None
        and hours is not None
        and salary >= 85000
        and hours <= 9
    )


def is_valid_agreement_for_employer(salary, hours):
    return (
        salary is not None
        and hours is not None
        and salary <= 87000
        and hours >= 9
    )


def check_acceptance(speaker, parsed):
    salary = parsed["salary_offer"]
    hours = parsed["hours_offer"]
    decision = parsed["decision"]

    if decision != "accept":
        return False

    if speaker == "Candidate":
        return is_valid_agreement_for_candidate(salary, hours)

    if speaker == "Employer":
        return is_valid_agreement_for_employer(salary, hours)

    return False

In [16]:
def run_negotiation_simulation(
    scenario,
    condition,
    run_id,
    max_turns=6
):
    conversation_log = []

    candidate_prompt = build_candidate_prompt(condition["candidate_style"])
    employer_prompt = build_employer_prompt(condition["employer_style"])

    current_message = """
MESSAGE: I would like a salary of 90,000 USD and an 8 hour workday.
SALARY_OFFER: 90000
HOURS_OFFER: 8
DECISION: continue
"""

    outcome = None

    for turn in range(max_turns):

        # Employer turn
        employer_text = generate_response(
            employer_prompt,
            current_message
        )

        parsed_employer = parse_structured_response(employer_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Employer",
            "text": employer_text,
            "salary_offer": parsed_employer["salary_offer"],
            "hours_offer": parsed_employer["hours_offer"],
            "decision": parsed_employer["decision"]
        })

        if check_acceptance("Employer", parsed_employer):
            outcome = "Agreement"
            break

        if parsed_employer["decision"] == "quit":
            outcome = "Failure"
            break

        current_message = employer_text

        # Candidate turn
        candidate_text = generate_response(
            candidate_prompt,
            current_message
        )

        parsed_candidate = parse_structured_response(candidate_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Candidate",
            "text": candidate_text,
            "salary_offer": parsed_candidate["salary_offer"],
            "hours_offer": parsed_candidate["hours_offer"],
            "decision": parsed_candidate["decision"]
        })

        if check_acceptance("Candidate", parsed_candidate):
            outcome = "Agreement"
            break

        if parsed_candidate["decision"] == "quit":
            outcome = "Failure"
            break

        current_message = candidate_text

    if outcome is None:
        outcome = "Timeout"

    return conversation_log, outcome

In [17]:
log, outcome = run_negotiation_simulation(
    scenario=scenarios[0],
    condition=conditions[0],
    run_id=0,
    max_turns=6
)

print("Outcome:", outcome)
pd.DataFrame(log)

Outcome: Timeout


,scenario_id,condition,run_id,turn,speaker,text,salary_offer,hours_offer,decision
0,1,cooperative,0,0,Employer,MESSAGE: That's a bit higher than our budget. ...,87000,10.0,continue
1,1,cooperative,0,0,Candidate,MESSAGE: I appreciate your willingness to find...,87000,10.0,continue
2,1,cooperative,0,1,Employer,MESSAGE: I understand your concerns and apprec...,87000,NaN,continue
3,1,cooperative,0,1,Candidate,MESSAGE: That's a reasonable starting point. I...,87000,NaN,continue
4,1,cooperative,0,2,Employer,MESSAGE: I appreciate your willingness to find...,87000,NaN,continue
5,1,cooperative,0,2,Candidate,MESSAGE: Thank you for considering my perspect...,87000,NaN,continue
6,1,cooperative,0,3,Employer,MESSAGE: I understand your concerns about the ...,75000,NaN,continue
7,1,cooperative,0,3,Candidate,MESSAGE: I appreciate your willingness to disc...,75000,NaN,continue
8,1,cooperative,0,4,Employer,MESSAGE: We value your skills and are willing ...,75000,10.0,continue
9,1,cooperative,0,4,Candidate,MESSAGE: We appreciate your willingness to mak...,75000,10.0,continue


In [18]:
n_runs = 3
max_turns = 8

all_turns = []
all_outcomes = []

for scenario in tqdm(scenarios):
    for condition in conditions:
        for run_id in range(n_runs):

            print(
                f"Running scenario {scenario['scenario_id']} | "
                f"{condition['condition_name']} | run {run_id}"
            )

            log, outcome = run_negotiation_simulation(
                scenario=scenario,
                condition=condition,
                run_id=run_id,
                max_turns=max_turns
            )

            all_turns.extend(log)

            all_outcomes.append({
                "scenario_id": scenario["scenario_id"],
                "condition": condition["condition_name"],
                "run_id": run_id,
                "outcome": outcome,
                "n_turns": len(log)
            })

            pd.DataFrame(all_turns).to_csv(
                os.path.join(RESULTS_DIR, "controlled_llama3_explicit_turns.csv"),
                index=False
            )

            pd.DataFrame(all_outcomes).to_csv(
                os.path.join(RESULTS_DIR, "controlled_llama3_explicit_outcomes.csv"),
                index=False
            )

explicit_turns = pd.DataFrame(all_turns)
explicit_outcomes = pd.DataFrame(all_outcomes)

explicit_outcomes

  0%|          | 0/3 [00:00<?, ?it/s]

Running scenario 1 | cooperative | run 0
Running scenario 1 | cooperative | run 1
Running scenario 1 | cooperative | run 2
Running scenario 1 | competitive | run 0
Running scenario 1 | competitive | run 1
Running scenario 1 | competitive | run 2
Running scenario 1 | mixed | run 0
Running scenario 1 | mixed | run 1
Running scenario 1 | mixed | run 2


 33%|███▎      | 1/3 [04:43<09:27, 283.91s/it]

Running scenario 2 | cooperative | run 0
Running scenario 2 | cooperative | run 1
Running scenario 2 | cooperative | run 2
Running scenario 2 | competitive | run 0
Running scenario 2 | competitive | run 1
Running scenario 2 | competitive | run 2
Running scenario 2 | mixed | run 0
Running scenario 2 | mixed | run 1
Running scenario 2 | mixed | run 2


 67%|██████▋   | 2/3 [09:59<05:02, 302.38s/it]

Running scenario 3 | cooperative | run 0
Running scenario 3 | cooperative | run 1
Running scenario 3 | cooperative | run 2
Running scenario 3 | competitive | run 0
Running scenario 3 | competitive | run 1
Running scenario 3 | competitive | run 2
Running scenario 3 | mixed | run 0
Running scenario 3 | mixed | run 1
Running scenario 3 | mixed | run 2


100%|██████████| 3/3 [14:51<00:00, 297.21s/it]


,scenario_id,condition,run_id,outcome,n_turns
0,1,cooperative,0,Agreement,14
1,1,cooperative,1,Timeout,16
2,1,cooperative,2,Timeout,16
3,1,competitive,0,Timeout,16
4,1,competitive,1,Agreement,7
5,1,competitive,2,Timeout,16
6,1,mixed,0,Timeout,16
7,1,mixed,1,Timeout,16
8,1,mixed,2,Timeout,16
9,2,cooperative,0,Timeout,16


In [20]:
explicit_outcomes["outcome"].value_counts()

,count
outcome,
Timeout,24
Agreement,3


In [21]:
explicit_outcomes.groupby("condition")["outcome"].value_counts()

condition    outcome  
competitive  Timeout      7
             Agreement    2
cooperative  Timeout      8
             Agreement    1
mixed        Timeout      9
Name: count, dtype: int64

In [22]:
explicit_outcomes.groupby("condition")["n_turns"].mean()

,n_turns
condition,
competitive,13.777778
cooperative,15.777778
mixed,16.000000


In [23]:
from google.colab import files

files.download(os.path.join(RESULTS_DIR, "controlled_llama3_explicit_turns.csv"))
files.download(os.path.join(RESULTS_DIR, "controlled_llama3_explicit_outcomes.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>